# End-to-End Sentiment Analysis Pipeline


This notebook merges the exploration, feature extraction, baseline modeling, and BERT workflows into a single cohesive pipeline, terminating with a Gradio web interface.


In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/dongrelaxman/amazon-reviews-dataset/Amazon_Reviews.csv


## 1. Installations and Imports


In [2]:
!pip install -q pandas numpy scikit-learn gensim gradio transformers torch

import pandas as pd
import numpy as np
import re
import pickle
import os
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
import gensim
import gradio as gr
import warnings
warnings.filterwarnings('ignore')

## 2. Data Loading and Preprocessing


In [5]:
# Load Data
# Note: Adjust the path as per your environment

data = pd.read_csv('/kaggle/input/datasets/dongrelaxman/amazon-reviews-dataset/Amazon_Reviews.csv', engine='python')


# Preprocessing
def preprocess_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    return text

if 'Rating' in data.columns:
    data['Rating'] = data['Rating'].astype(str).str.extract(r'(\d)').astype(float)
    data = data.dropna(subset=['Rating'])
    data['Sentiment'] = data['Rating'].apply(lambda x: 1 if x >= 4 else (0 if x <= 2 else None))
    data = data.dropna(subset=['Sentiment'])
    data['Sentiment'] = data['Sentiment'].astype(int)
    
if 'Review Title' in data.columns and 'Review Text' in data.columns:
    data['Full_Review'] = data['Review Title'].fillna('') + " " + data['Review Text'].fillna('')
elif 'Review Text' in data.columns:
    data['Full_Review'] = data['Review Text'].fillna('')
else:
    data['Full_Review'] = data.iloc[:, 0].astype(str)

data['Clean_Review'] = data['Full_Review'].apply(preprocess_text)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(data['Clean_Review'], data['Sentiment'], test_size=0.2, random_state=42)
print(f"Training samples: {len(X_train)}, Testing samples: {len(X_test)}")

Training samples: 16136, Testing samples: 4034


## 3. Feature Extraction


In [6]:
# TF-IDF
tfidf_vectorizer = TfidfVectorizer(max_features=5000)
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

# Word2Vec
# Tokenize sentences for Word2Vec
sentences_train = [text.split() for text in X_train]
sentences_test = [text.split() for text in X_test]

w2v_model = gensim.models.Word2Vec(sentences_train, vector_size=100, window=5, min_count=1, workers=4)

def get_sentence_vector(words, model):
    valid_words = [word for word in words if word in model.wv]
    if valid_words:
        return np.mean(model.wv[valid_words], axis=0)
    else:
        return np.zeros(model.vector_size)

X_train_w2v = np.array([get_sentence_vector(words, w2v_model) for words in sentences_train])
X_test_w2v = np.array([get_sentence_vector(words, w2v_model) for words in sentences_test])

## 4. Model Training and Evaluation


In [7]:
# Train LR on TF-IDF
lr_tfidf = LogisticRegression(max_iter=1000)
lr_tfidf.fit(X_train_tfidf, y_train)
preds_tfidf = lr_tfidf.predict(X_test_tfidf)
print("TF-IDF Model Accuracy:", accuracy_score(y_test, preds_tfidf))
print(classification_report(y_test, preds_tfidf))

# Train LR on Word2Vec
lr_w2v = LogisticRegression(max_iter=1000)
lr_w2v.fit(X_train_w2v, y_train)
preds_w2v = lr_w2v.predict(X_test_w2v)
print("Word2Vec Model Accuracy:", accuracy_score(y_test, preds_w2v))
print(classification_report(y_test, preds_w2v))

TF-IDF Model Accuracy: 0.9536440257808627
              precision    recall  f1-score   support

           0       0.95      0.99      0.97      2870
           1       0.96      0.87      0.92      1164

    accuracy                           0.95      4034
   macro avg       0.96      0.93      0.94      4034
weighted avg       0.95      0.95      0.95      4034

Word2Vec Model Accuracy: 0.9248884481903817
              precision    recall  f1-score   support

           0       0.93      0.97      0.95      2870
           1       0.91      0.82      0.86      1164

    accuracy                           0.92      4034
   macro avg       0.92      0.90      0.91      4034
weighted avg       0.92      0.92      0.92      4034



## 5. Saving and Loading Models


In [8]:
os.makedirs("models", exist_ok=True)

# Save TF-IDF and Model
with open("models/tfidf_vectorizer.pkl", "wb") as f:
    pickle.dump(tfidf_vectorizer, f)
with open("models/lr_tfidf.pkl", "wb") as f:
    pickle.dump(lr_tfidf, f)

# Save Word2Vec and Model
w2v_model.save("models/w2v_model.gensim")
with open("models/lr_w2v.pkl", "wb") as f:
    pickle.dump(lr_w2v, f)

print("Models saved successfully in the 'models' directory.")

Models saved successfully in the 'models' directory.


## 6. Gradio Application


In [9]:
# Load models for inference
with open("models/tfidf_vectorizer.pkl", "rb") as f:
    loaded_tfidf_vec = pickle.load(f)
with open("models/lr_tfidf.pkl", "rb") as f:
    loaded_lr_tfidf = pickle.load(f)

loaded_w2v_model = gensim.models.Word2Vec.load("models/w2v_model.gensim")
with open("models/lr_w2v.pkl", "rb") as f:
    loaded_lr_w2v = pickle.load(f)

def predict_sentiment(text, model_choice):
    clean_text = preprocess_text(text)
    
    if model_choice == "TF-IDF":
        vec = loaded_tfidf_vec.transform([clean_text])
        pred = loaded_lr_tfidf.predict(vec)[0]
    else:
        words = clean_text.split()
        vec = get_sentence_vector(words, loaded_w2v_model).reshape(1, -1)
        pred = loaded_lr_w2v.predict(vec)[0]
        
    return "Positive 🟢" if pred == 1 else "Negative 🔴"

# Gradio Interface
iface = gr.Interface(
    fn=predict_sentiment,
    inputs=[
        gr.Textbox(lines=3, placeholder="Enter a review here...", label="Review Text"),
        gr.Radio(["TF-IDF", "Word2Vec"], value="TF-IDF", label="Select Feature Extraction Method")
    ],
    outputs=gr.Text(label="Predicted Sentiment"),
    title="Sentiment Analysis Classifier",
    description="Enter a review to predict whether the sentiment is Positive or Negative using Logistic Regression on TF-IDF or Word2Vec features."
)

# Launch the app
iface.launch(share=True)

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://0a136a04b486c3ae79.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Created dataset file at: .gradio/flagged/dataset1.csv
